In [ ]:
import numpy as np
from affine import Affine
from scipy.ndimage import gaussian_filter
from rasterio.features import rasterize, geometry_mask
import geopandas as gpd
import matplotlib.pyplot as plt
import pandas as pd
import matplotlib.patches as mpatches
import os
import fiona
import rasterio
from rasterio.enums import MergeAlg
from sklearn.preprocessing import MinMaxScaler
from shapely.geometry import Point
import libpysal
import esda
import rasterstats
import warnings
warnings.filterwarnings("ignore")
import sys
import importlib

sys.path.insert(0, os.path.abspath("../.."))


In [ ]:
import utils.config
import utils.functions
importlib.reload(utils.config)
importlib.reload(utils.functions)

from utils.config import *
from utils.functions import *

In [ ]:
#Read the files
index_walkability = gpd.read_parquet(f'{output_step3_path}/step3_index.parquet')
index_walkability = index_walkability.to_crs(operation_crs)

carreau_200 = gpd.read_file(f'{output_step3_path}/step3_aggregated_index_carreau200.gpkg')
carreau_200 = carreau_200.to_crs(operation_crs)

zones_girec = gpd.read_file(f'{output_step3_path}/step3_aggregated_index_girec.gpkg')
zones_girec = zones_girec.to_crs(operation_crs)


user_stat  = pd.read_csv(f'{input_file_path_PL}/241120_user_statistics.csv')

agglo_GG  = gpd.read_file(f'{input_file_path}/network_agreg/AGGLO_PERIMETRE_AVEC_LAC-SHP/AGGLO_PERIMETRE_AVEC_LAC.shp')
agglo_GG = agglo_GG.to_crs(operation_crs)

respondants_wave1 = pd.read_csv(f'{output_file_path_PL_wave1}respondants_all_wave1.csv')

# DATA LOADING

In [ ]:
# ─── Comptage brut du fichier original (sans filtres) ─────────────────────────
layers = fiona.listlayers(f'{input_file_path_PL}241120_legs.gpkg')

total_legs = 0
all_user_ids = set()

for layer in layers:
    gdf_raw = gpd.read_file(
        f'{input_file_path_PL}241120_legs.gpkg',
        layer=layer,
        columns=["user_id_fors"]  # charge uniquement la colonne nécessaire
    )
    total_legs += len(gdf_raw)
    all_user_ids.update(gdf_raw["user_id_fors"].dropna().unique())

print(f"Fichier original : {total_legs} legs / {len(all_user_ids)} utilisateurs")

## LEGS_GG_ALL (tous les déplacements dans Genève, tous modes confondus)

In [ ]:
# ─── Préparation du périmètre géographique Grand Genève (agglo_GG) ────────────
from shapely.geometry import LineString, MultiLineString

# On fusionne toutes les géométries de agglo_GG en une seule pour le test
# d'appartenance (au cas où le shapefile contient plusieurs polygones/îles).
try:
    agglo_GG_geom = agglo_GG.union_all()          # geopandas >= 0.14
except AttributeError:
    agglo_GG_geom = agglo_GG.unary_union           # fallback anciennes versions


def get_start_end_points(geom):
    """Retourne les points de début et de fin d'une géométrie de leg (LineString)."""
    if geom is None or geom.is_empty:
        return None, None
    if isinstance(geom, LineString):
        coords = list(geom.coords)
        return Point(coords[0]), Point(coords[-1])
    if isinstance(geom, MultiLineString):
        parts = list(geom.geoms)
        return Point(list(parts[0].coords)[0]), Point(list(parts[-1].coords)[-1])
    # cas de repli (ex: géométrie déjà en Point)
    return geom, geom


def compute_intra_GG(gdf, perimeter_geom, perimeter_crs):
    """
    intra_GG = 1 si le point de départ ET le point d'arrivée du leg
    sont contenus dans le périmètre Grand Genève (agglo_GG).
    """
    gdf_proj = gdf.to_crs(perimeter_crs)

    starts, ends = zip(*gdf_proj.geometry.map(get_start_end_points))
    starts = gpd.GeoSeries(starts, crs=perimeter_crs)
    ends   = gpd.GeoSeries(ends,   crs=perimeter_crs)

    start_in = starts.intersects(perimeter_geom)
    end_in   = ends.intersects(perimeter_geom)

    return (start_in & end_in).fillna(False).astype(int).values


# ─── Chargement tous modes, filtres qualité + sélection géographique GG ───────
layers     = fiona.listlayers(f'{input_file_path_PL}241120_legs.gpkg')
all_layers = []

for layer in layers:
    where_clause = """
        extreme99_length_mode = 0 AND
        extreme98_length_mode = 0 AND
        usr_w_constant_bad_signal = 0

    """

    gdf = gpd.read_file(
        f'{input_file_path_PL}241120_legs.gpkg',
        layer=layer,
        where=where_clause
    )

    if len(gdf) > 0:
        gdf["intra_GG"] = compute_intra_GG(gdf, agglo_GG_geom, agglo_GG.crs)
        gdf = gdf[gdf["intra_GG"] == 1].copy()

    print(f"── {layer:<25} selected: {len(gdf):>6}")
    if len(gdf) > 0:
        gdf["source_layer"] = layer
        all_layers.append(gdf)

legs_GG_all = gpd.GeoDataFrame(
    pd.concat(all_layers, ignore_index=True),
    crs="EPSG:4326"
)
del all_layers

print(f"\nlegs_all_GG : {len(legs_GG_all)} legs / {legs_GG_all['user_id_fors'].nunique()} utilisateurs")


In [ ]:
len(layers)

In [ ]:
print(legs_GG_all.dtypes.to_string())

In [ ]:
from IPython.display import display, HTML
display(HTML(legs_GG_all.drop(columns='geometry').head(10).to_html()))

In [ ]:
fig, ax = plt.subplots(figsize=(10, 10))
fig.suptitle("legs_GG_all", fontsize=13, fontweight="bold")

legs_GG_all.to_crs(2056).plot(ax=ax, color="#4C72B0", linewidth=0.3, alpha=0.3)
agglo_GG.boundary.plot(ax=ax, color='black', linewidth=1.5)
ax.set_axis_off()
plt.tight_layout()
plt.show()

### AJOUT COL LEGSGGALL

In [ ]:
# ─── Merges socio-démo ────────────────────────────────────────────────────────

# Merge user_stat
cols_to_add = ["user_id_fors"] + [
    col for col in user_stat.columns
    if col not in legs_GG_all.columns
]
legs_GG_all = legs_GG_all.merge(user_stat[cols_to_add], on="user_id_fors", how="left")

# Merge respondants_wave1 (abonnements + revenus)
abo_cols_1  = [f'Q10_{i}_R' for i in range(1, 10)]
abo_cols_2  = [f'Q11_{i}_R' for i in range(1, 10)]
revenu_cols = ["Q120", "Q121"]
physical_cond_col = ["Q108_new"]
compo_menage = [f'Q119_{i}' for i in range(1, 10)]
all_cols    = abo_cols_1 + abo_cols_2 + revenu_cols + physical_cond_col + compo_menage
cols_wave1  = [c for c in all_cols if c in respondants_wave1.columns and c not in legs_GG_all.columns]

if cols_wave1:
    legs_GG_all = legs_GG_all.merge(
        respondants_wave1[["id"] + cols_wave1].rename(columns={"id": "user_id_fors"}),
        on="user_id_fors",
        how="left"
    )

# Merge trip/journey
leg_trip_journey = pd.read_csv(f'{input_file_path_PL}241120_map_track_trip_journey.csv')
legs_GG_all = legs_GG_all.merge(leg_trip_journey, on="leg_id", how="left")

In [ ]:
# ─── Regroupement age 60+ ─────────────────────────────────────────────────────
legs_GG_all["age_fr_grouped"] = legs_GG_all["age_fr"].replace({
    "60-74 ans"      : "60 ans +",
    "75 ans et plus" : "60 ans +",
})

# ─── Vérification ─────────────────────────────────────────────────────────────
print(legs_GG_all[["age_fr", "age_fr_grouped"]].drop_duplicates().sort_values("age_fr_grouped").to_string(index=False))

In [ ]:
# ─── Vérifier les modes disponibles ──────────────────────────────────────────
print("── Modes disponibles ────────────────────────────────")
print(legs_GG_all["mode"].value_counts())

In [ ]:
# ─── Variables calculées ──────────────────────────────────────────────────────
legs_GG_all = legs_GG_all.assign(
    length_weighted = lambda x: x['length_leg'] * x['wgt_cant_trim_gps'] * x['intra_GG'],
)

# ─── Mapping mode → groupe ────────────────────────────────────────────────────
mode_to_group = {
    mode: group
    for group, modes in mode_groups.items()
    for mode in modes
}
legs_GG_all["mode_group"] = legs_GG_all["mode"].map(mode_to_group).fillna("other")

# ─── Stats globales par user ──────────────────────────────────────────────────
_stats = legs_GG_all.groupby('user_id_fors').agg(
    n_legs_total = ('leg_id',          'count'),
    dist_total   = ('length_weighted', 'sum'),
    n_days_GG    = ('legs_date',       'nunique'),
).reset_index()

# ─── Distance et legs par mode par user ───────────────────────────────────────
_dist_by_mode = (
    legs_GG_all
    .groupby(["user_id_fors", "mode_group"])["length_weighted"]
    .sum()
    .unstack(fill_value=0)
    .add_prefix("dist_")
    .reset_index()
)

_legs_by_mode = (
    legs_GG_all
    .groupby(["user_id_fors", "mode_group"])["leg_id"]
    .count()
    .unstack(fill_value=0)
    .add_prefix("n_legs_")
    .reset_index()
)

# ─── Stats spécifiques à la marche ────────────────────────────────────────────
_walk_stats = (
    legs_GG_all[legs_GG_all["mode_group"] == "walk"]
    .groupby("user_id_fors")
    .agg(
        n_walk_days = ('legs_date', 'nunique'),  # ← jours avec au moins 1 leg marche
    )
    .reset_index()
)

# ─── Merger tout ──────────────────────────────────────────────────────────────
_stats = _stats.merge(_dist_by_mode,  on="user_id_fors", how="left")
_stats = _stats.merge(_legs_by_mode,  on="user_id_fors", how="left")
_stats = _stats.merge(_walk_stats,    on="user_id_fors", how="left")

# ─── Fillna ───────────────────────────────────────────────────────────────────
for group in mode_groups.keys():
    for prefix in ["dist_", "n_legs_"]:
        col = f"{prefix}{group}"
        if col in _stats.columns:
            _stats[col] = _stats[col].fillna(0)
_stats["n_walk_days"] = _stats["n_walk_days"].fillna(0)

# ─── Parts modales et distances journalières par mode ─────────────────────────
for group in mode_groups.keys():
    dist_col  = f"dist_{group}"
    legs_col  = f"n_legs_{group}"

    if dist_col in _stats.columns:
        _stats[f"dist_{group}_per_day"]  = _stats[dist_col] / _stats["n_days_GG"]
        _stats[f"share_dist_{group}"]    = _stats[dist_col] / _stats["dist_total"]
        _stats[f"share_legs_{group}"]    = _stats[legs_col] / _stats["n_legs_total"]

# ─── Métriques spécifiques marche ─────────────────────────────────────────────
_stats["walk_share_dist"]         = _stats["dist_walk"]   / _stats["dist_total"]
_stats["walk_share_legs"]         = _stats["n_legs_walk"] / _stats["n_legs_total"]
_stats["dist_walk_per_day"]       = _stats["dist_walk"]   / _stats["n_days_GG"]
_stats["dist_walk_per_walk_day"]  = np.where(
    _stats["n_walk_days"] > 0,
    _stats["dist_walk"] / _stats["n_walk_days"],
    np.nan
)


_stats["mean_walk_legs_per_day"] = _stats["n_legs_walk"] / _stats["n_days_GG"]



# ─── Flag revenu ──────────────────────────────────────────────────────────────
legs_GG_all['a_repondu_revenu'] = (
    (legs_GG_all['Q120'].notna() & (legs_GG_all['Q120'] != 10)) |
    (legs_GG_all['Q121'].notna() & (legs_GG_all['Q121'] != 12))
)

# ─── Merger sur legs_GG_all ───────────────────────────────────────────────────
legs_GG_all = legs_GG_all.merge(_stats, on='user_id_fors', how='left')
del _stats, _dist_by_mode, _legs_by_mode, _walk_stats

# ─── Vérification ─────────────────────────────────────────────────────────────
print(f"legs_GG_all enrichi : {len(legs_GG_all)} legs / {legs_GG_all['user_id_fors'].nunique()} users")
print(f"Colonnes            : {len(legs_GG_all.columns)}")
print(f"\n── Nouvelles colonnes ───────────────────────────────")
for group in mode_groups.keys():
    print(f"  dist_{group:<25} | dist_{group}_per_day | share_dist_{group} | share_legs_{group}")
print(f"  dist_walk_per_walk_day | mean_walk_legs_per_day | n_walk_days")

In [ ]:
# ─── Variables temporelles ────────────────────────────────────────────────────
legs_GG_all["started_at_local"]  = pd.to_datetime(legs_GG_all["started_at_in_timezone"], utc=True).dt.tz_convert("Europe/Zurich")
legs_GG_all["finished_at_local"] = pd.to_datetime(legs_GG_all["finished_at_in_timezone"], utc=True).dt.tz_convert("Europe/Zurich")

legs_GG_all["hour"]            = legs_GG_all["started_at_local"].dt.hour
legs_GG_all["hour_end"]        = legs_GG_all["finished_at_local"].dt.hour
legs_GG_all["time_slot"]       = legs_GG_all["started_at_local"].dt.floor("15min").dt.strftime("%H:%M")
legs_GG_all["legs_date"]       = legs_GG_all["started_at_local"].dt.date
legs_GG_all["day_of_week"]     = legs_GG_all["started_at_local"].dt.day_name()
legs_GG_all["day_of_week_num"] = legs_GG_all["started_at_local"].dt.dayofweek
legs_GG_all["duration_min"]    = legs_GG_all["duration"] / 60

legs_GG_all['duration_min_weighted'] = (
    legs_GG_all['duration_min'] * legs_GG_all['wgt_cant_trim_gps'] * legs_GG_all['intra_GG']
)

# ─── Durée totale de marche par utilisateur ───────────────────────────────────
_dur_walk = (
    legs_GG_all[legs_GG_all['mode'] == 'Mode::Walk']
    .groupby('user_id_fors')['duration_min_weighted']
    .sum()
    .rename('total_duration_walk_min')
    .reset_index()
)

legs_GG_all = legs_GG_all.merge(_dur_walk, on='user_id_fors', how='left')
del _dur_walk

legs_GG_all['total_duration_walk_min']    = legs_GG_all['total_duration_walk_min'].fillna(0)
legs_GG_all['mean_walk_duration_per_day'] = (
    legs_GG_all['total_duration_walk_min'] / legs_GG_all['n_days_GG']
)

print(f"Vérification durée moyenne : {legs_GG_all.drop_duplicates('user_id_fors')['mean_walk_duration_per_day'].median():.1f} min/jour")

In [ ]:
# ─── Flag timestamp_error (vitesse > 30 km/h sur legs de marche) ─────────────
WALK_SPEED_MAX_KMH = 30

legs_GG_all["speed_kmh"] = (
    legs_GG_all["length_leg"] / (legs_GG_all["duration_min"] / 60) / 1000
)

legs_GG_all["timestamp_error"] = (
    (legs_GG_all["mode"] == "Mode::Walk") &
    (legs_GG_all["speed_kmh"] > WALK_SPEED_MAX_KMH)
).astype(int)

print(f"Legs flaggés timestamp_error : {legs_GG_all['timestamp_error'].sum()} "
      f"({legs_GG_all['timestamp_error'].mean()*100:.2f}% des legs)")
print(f"Utilisateurs concernés : {legs_GG_all[legs_GG_all['timestamp_error']==1]['user_id_fors'].nunique()}")

In [ ]:
# ─── Insérer les labels juste après les colonnes Q120 et Q121 ─────────────────
idx_Q120 = legs_GG_all.columns.get_loc('Q120') + 1
legs_GG_all.insert(idx_Q120, 'Q120_label', legs_GG_all['Q120'].map(income_labels_Q120))

idx_Q121 = legs_GG_all.columns.get_loc('Q121') + 1
legs_GG_all.insert(idx_Q121, 'Q121_label', legs_GG_all['Q121'].map(income_labels_Q121))

In [ ]:
from IPython.display import display, HTML
display(HTML(legs_GG_all.drop(columns='geometry').head(10).to_html()))

In [ ]:
# ─── Classification revenu ménage selon Q120 (ref. GE annuel / 12) ────────────
# Q1 = 63'281 / 12 = 5'273 CHF/mois
# Q3 = 172'719 / 12 = 14'393 CHF/mois
# Source : Panel Lémanique, statistiques GE

def classify_income_Q120(q120):
    if pd.isna(q120):
        return np.nan
    q120 = int(q120)
    if q120 in [1, 2, 3]:             return labels_all_groups["tres_modeste"]  # < 6'000 CHF/mois
    elif q120 in [4, 5, 6, 7]:        return labels_all_groups["modeste"]       # 6'001 - 14'000 CHF/mois
    elif q120 in [8, 9]:              return labels_all_groups["aise"]           # > 14'001 CHF/mois
    else:                             return np.nan                              # 10 = Ne sait pas

legs_GG_all['income_class_GE'] = legs_GG_all['Q120'].apply(classify_income_Q120)


# ─── Classification revenu ménage selon Q121 (ref. INSEE France 2018 / 12) ────
# Q1 ≈ 25760/12 = 2146 €/mois  → seuil entre codes 2 et 3 --> on dit que les code 1 et 2 = sous Q1
# Q3 ≈ 42480/12 = 3540 €/mois  → seuil entre codes 3 et 4 --> tout ce qui est au dessus du code 4 = aisé
# Source : INSEE, Revenus fiscaux et sociaux 2018 (France métropolitaine)

def classify_income_Q121(q121):
    if pd.isna(q121):
        return np.nan
    q121 = int(q121)
    if q121 in [1, 2]:                     return labels_all_groups["tres_modeste"]  # < 2 000 €/mois
    elif q121 in [3, 4]:                   return labels_all_groups["modeste"]       # 2 001 - 4 000 €/mois
    elif q121 in [5, 6, 7, 8, 9, 10, 11]: return labels_all_groups["aise"]          # > 4 001 €/mois
    else:                                  return np.nan                             # 12 = Ne sait pas / Préfère ne pas répondre

legs_GG_all['income_class_FR'] = legs_GG_all['Q121'].apply(classify_income_Q121)


# ─── Colonne unifiée : GE prioritaire, FR en complément ───────────────────────
legs_GG_all['income_class'] = legs_GG_all['income_class_GE'].fillna(legs_GG_all['income_class_FR'])


# ─── Vérification ─────────────────────────────────────────────────────────────
_u = legs_GG_all.drop_duplicates('user_id_fors')  # ← après le fillna

both_filled = _u['income_class_GE'].notna() & _u['income_class_FR'].notna()
n_both = both_filled.sum()
if n_both == 0:
    print("✓ Aucun répondant n'a rempli Q120 et Q121 simultanément\n")
else:
    print(f"⚠️  {n_both} répondant(s) ont rempli Q120 ET Q121 — à vérifier !")
    print(_u[both_filled][['user_id_fors', 'income_class_GE', 'income_class_FR']])

for col in ['income_class_GE', 'income_class_FR', 'income_class']:
    print(f"── {col} {'─' * (40 - len(col))}")
    for cat in income_class_order:
        n   = (_u[col] == cat).sum()
        pct = n / _u[col].notna().sum() * 100
        print(f"  {cat:<25} : {n:>4} ({pct:.1f}%)")
    print(f"  {'NaN':<25} : {_u[col].isna().sum()}\n")

del _u

In [ ]:
# ─── Q10 : abonnement TP principal ────────────────────────────────────────────
# Priorité : AG > Léman Pass > Abo communautaire > Demi-tarif > Abo parcours > Autre > Aucun
def get_main_tp(row):
    if row.get('Q10_1_R') == 1: return 'AG'
    if row.get('Q10_6_R') == 1: return 'Léman Pass'
    if row.get('Q10_7_R') == 1: return 'Abo communautaire'
    if row.get('Q10_2_R') == 1: return 'Demi-tarif'
    if row.get('Q10_5_R') == 1: return 'Abo parcours CFF'
    if row.get('Q10_3_R') == 1: return 'Abo forfaitaire SNCF'
    if row.get('Q10_4_R') == 1: return 'Carte réduction SNCF'
    if row.get('Q10_8_R') == 1: return 'Autre'
    if row.get('Q10_9_R') == 1: return 'Aucun abonnement'
    return np.nan

# ─── Q10 : niveau TP (pour captivité) ────────────────────────────────────────
# 0 = aucun, 1 = partiel, 2 = fort
def get_tp_level(row):
    if row.get('Q10_1_R') == 1: return 2   # AG
    if row.get('Q10_6_R') == 1: return 2   # Léman Pass -> https://www.lemanpass.com/tarifs/ (à définir, mais permet de voyager librement dans les zones choisies)
    if row.get('Q10_7_R') == 1: return 2   # Abo communautaire -> voyager en illimité dans les zones choisies
    if row.get('Q10_2_R') == 1: return 1   # Demi-tarif
    if row.get('Q10_5_R') == 1: return 1   # Abo parcours -> voyager en illimité sur un trajet en particulier
    if row.get('Q10_9_R') == 1: return 0   # Aucun
    return np.nan

# ─── Insérer après Q10_9_R ────────────────────────────────────────────────────
tp_cols = [c for c in tp_labels_Q10.keys() if c in legs_GG_all.columns]
if tp_cols:
    idx_tp = legs_GG_all.columns.get_loc(tp_cols[-1]) + 1
    legs_GG_all.insert(idx_tp,     'main_tp_abo', legs_GG_all[tp_cols].apply(get_main_tp,  axis=1))
    legs_GG_all.insert(idx_tp + 1, 'tp_level',    legs_GG_all[tp_cols].apply(get_tp_level, axis=1).astype('Int64'))

# ─── Q11 : abonnement mobilité principal ──────────────────────────────────────
# Priorité : Vignette/Télépéage > Autopartage > P+R > B+R > VLS > Aucun
def get_main_mobility(row):
    if row.get('Q11_1_R') == 1: return 'Vignette autoroute'
    if row.get('Q11_2_R') == 1: return 'Badge télépéage'
    if row.get('Q11_5_R') == 1: return 'Autopartage (perso)'
    if row.get('Q11_6_R') == 1: return 'Autopartage (employeur)'
    if row.get('Q11_3_R') == 1: return 'Abo P+R'
    if row.get('Q11_4_R') == 1: return 'Abo B+R (bike and ride)'
    if row.get('Q11_7_R') == 1: return 'VLS (perso)'
    if row.get('Q11_8_R') == 1: return 'VLS (employeur)'
    if row.get('Q11_9_R') == 1: return 'Aucun abo'
    return np.nan

# ─── Q11 : niveau motorisation (pour captivité) ───────────────────────────────
# 0 = aucun, 1 = mobilité douce/partagée, 2 = voiture
def get_mobility_level(row):
    if row.get('Q11_1_R') == 1: return 2   # Vignette → voiture régulière
    if row.get('Q11_2_R') == 1: return 2   # Télépéage → voiture régulière
    if row.get('Q11_5_R') == 1: return 1   # Autopartage perso
    if row.get('Q11_6_R') == 1: return 1   # Autopartage employeur
    if row.get('Q11_3_R') == 1: return 1   # P+R
    if row.get('Q11_4_R') == 1: return 1   # B+R
    if row.get('Q11_7_R') == 1: return 1   # VLS perso
    if row.get('Q11_8_R') == 1: return 1   # VLS employeur
    if row.get('Q11_9_R') == 1: return 0   # Aucun
    return np.nan

# ─── Insérer après Q11_9_R ────────────────────────────────────────────────────
q11_cols = [c for c in tp_labels_Q11.keys() if c in legs_GG_all.columns]
if q11_cols:
    idx_q11 = legs_GG_all.columns.get_loc(q11_cols[-1]) + 1
    legs_GG_all.insert(idx_q11,     'main_mobility_abo', legs_GG_all[q11_cols].apply(get_main_mobility,  axis=1))
    legs_GG_all.insert(idx_q11 + 1, 'mobility_level',    legs_GG_all[q11_cols].apply(get_mobility_level, axis=1).astype('Int64'))

# ─── Vérification ─────────────────────────────────────────────────────────────
_u = legs_GG_all.drop_duplicates('user_id_fors')

print("── main_tp_abo ──────────────────────────────────────")
print(_u['main_tp_abo'].value_counts())
print(f"\n── tp_level ─────────────────────────────────────────")
print(_u['tp_level'].value_counts().sort_index())
print(f"\n── main_mobility_abo ────────────────────────────────")
print(_u['main_mobility_abo'].value_counts())
print(f"\n── mobility_level ───────────────────────────────────")
print(_u['mobility_level'].value_counts().sort_index())

del _u

In [ ]:
legs_GG_all['has_car'] = (
    legs_GG_all['car_in_HH_count']
    .apply(lambda x: 'Avec voiture' if pd.notna(x) and x > 0 else ('Sans voiture' if pd.notna(x) else np.nan))
    .map(car_fr_to_en)  # ← FR → EN
)

In [ ]:
# ─── Catégorie de marcheur selon intensité ────────────────────────────────────
legs_GG_all['walk_intensity'] = pd.cut(
    legs_GG_all['dist_walk_per_day'],
    bins   = [0, 100, 500, 1000, 2000, float('inf')],
    labels = ['Très faible (<100m)', 'Faible (100-500m)', 
              'Modéré (500m-1km)', 'Régulier (1-2km)', 'Intensif (>2km)'],
    right  = False
)

walk_intensity_order = legs_GG_all['walk_intensity'].cat.categories.tolist()

In [ ]:
physical_cond_label_order = list(physical_cond_labels.values())

idx_physical_cond = legs_GG_all.columns.get_loc('Q108_new') + 1
legs_GG_all.insert(idx_physical_cond, 'physical_cond_label',
                   legs_GG_all['Q108_new'].map(physical_cond_labels))

In [ ]:
print(legs_GG_all.dtypes.to_string())

In [ ]:
# ─── Colonnes Q119_1 à Q119_9 — activité professionnelle des membres du ménage ──
q119_cols = [f'Q119_{i}' for i in range(1, 10)]

# Vérifier lesquelles existent dans respondants_wave1
q119_available = [c for c in q119_cols if c in respondants_wave1.columns]
print(f"Colonnes Q119 disponibles : {q119_available}")
print(respondants_wave1[q119_available].head(10))
print("\nDistribution Q119_1 :")
print(respondants_wave1['Q119_1'].value_counts(dropna=False))

In [ ]:
legs_GG_all

In [ ]:
print(legs_GG_all.dtypes.to_string())

## USER_GG

In [ ]:
# ─── Dataframe permanent : une ligne par utilisateur ─────────────────────────
users_GG = legs_GG_all.drop_duplicates(subset='user_id_fors').copy()

print(f"users_GG : {len(users_GG)} utilisateurs")
print(f"Colonnes : {len(users_GG.columns)}")

# ─── Exploration des distributions ───────────────────────────────────────────
print("\n=== Distribution n_days_GG ===")
print(users_GG['n_days_GG'].describe())

print("\n=== Distribution walk_share_legs ===")
print(users_GG['walk_share_legs'].describe())

print("\n=== Distribution walk_share_dist ===")
print(users_GG['walk_share_dist'].describe())

print("\n=== Distribution dist_walk_per_day ===")
print(users_GG['dist_walk_per_day'].describe())

# ─── Sensibilité aux seuils ───────────────────────────────────────────────────
print("\n=== Sensibilité N_DAYS_MIN ===")
for n in [1, 3, 5, 7]:
    cnt = (users_GG['n_days_GG'] >= n).sum()
    print(f"  >= {n} jours : {cnt:>4} ({cnt/len(users_GG)*100:.1f}%)")

print("\n=== Sensibilité WALK_SHARE_MIN (sur n_days >= 3) ===")
_u3 = users_GG[users_GG['n_days_GG'] >= 3]
for seuil in [0.20, 0.30, 0.40, 0.50]:
    cnt = (_u3['walk_share_legs'] >= seuil).sum()
    print(f"  >= {seuil:.0%} : {cnt:>4} ({cnt/len(_u3)*100:.1f}%)")
del _u3

# ─── Visualisations ───────────────────────────────────────────────────────────
fig, axes = plt.subplots(2, 2, figsize=(14, 8))
fig.suptitle("Distributions — calibration des seuils", fontsize=14, fontweight='bold')

users_GG['n_days_GG'].hist(bins=30, ax=axes[0,0], color='#4C72B0')
axes[0,0].set_title('n_days_GG')
axes[0,0].axvline(x=3, color='red', linestyle='--', label='seuil=3')
axes[0,0].legend()

users_GG['walk_share_legs'].hist(bins=30, ax=axes[0,1], color='#DD8452')
axes[0,1].set_title('walk_share_legs')
axes[0,1].axvline(x=0.30, color='red', linestyle='--', label='seuil=0.30')
axes[0,1].legend()

users_GG['dist_walk_per_day'].hist(bins=50, ax=axes[1,0], color='#55A868')
axes[1,0].set_title('dist_walk_per_day (brut)')
axes[1,0].set_xlabel('mètres/jour')

np.log1p(users_GG['dist_walk_per_day']).hist(bins=50, ax=axes[1,1], color='#55A868')
axes[1,1].set_title('dist_walk_per_day (log)')
axes[1,1].set_xlabel('log(mètres/jour)')

plt.tight_layout()
plt.show()

### DATAFRAMES

In [ ]:
N_DAYS_MIN     = 0
WALK_SHARE_MIN = 0

# ─── Colonne is_walker dans legs_GG_all ───────────────────────────────────────
legs_GG_all['is_walker'] = (
    (legs_GG_all['n_days_GG']       >= N_DAYS_MIN) &
    (legs_GG_all['walk_share_legs'] >= WALK_SHARE_MIN)
).astype(int)

# ─── Sous-dataframes legs ─────────────────────────────────────────────────────
legs_GG_walk         = legs_GG_all[legs_GG_all['mode'] == 'Mode::Walk'].copy()
legs_GG_walk_regular = legs_GG_walk[legs_GG_walk['is_walker'] == 1].copy()

# ─── Sous-dataframes users (une ligne par utilisateur) ────────────────────────
users_GG_all              = legs_GG_all.drop_duplicates(subset='user_id_fors').copy()
users_GG_walk         = legs_GG_walk.drop_duplicates(subset='user_id_fors').copy()
users_GG_walk_regular = legs_GG_walk_regular.drop_duplicates(subset='user_id_fors').copy()

# ─── Résumé ───────────────────────────────────────────────────────────────────
n_base_legs = len(legs_GG_all)
n_base_users = len(users_GG_all)

print("─── legs ──────────────────────────────────────────────────")
print(f"legs_GG_all          : {len(legs_GG_all):>6} legs (référence)")
print(f"legs_GG_walk         : {len(legs_GG_walk):>6} legs ({len(legs_GG_walk)/n_base_legs*100:.1f}%)")
print(f"legs_GG_walk_regular : {len(legs_GG_walk_regular):>6} legs ({len(legs_GG_walk_regular)/n_base_legs*100:.1f}%)")

print("\n─── users ─────────────────────────────────────────────────")
print(f"users_GG_all              : {n_base_users:>4} utilisateurs (référence)")
print(f"users_GG_walk         : {len(users_GG_walk):>4} utilisateurs ({len(users_GG_walk)/n_base_users*100:.1f}%)")
print(f"users_GG_walk_regular : {len(users_GG_walk_regular):>4} utilisateurs ({len(users_GG_walk_regular)/n_base_users*100:.1f}%)")

print(f"\nColonnes identiques dans tous les dataframes : "
      f"{len(set(legs_GG_all.columns) - set(legs_GG_walk.columns)) == 0}")

In [ ]:
print("── Distribution Q120 (revenu ménage CHF) ────────────")
print(users_GG_walk_regular["Q120_label"].value_counts().reindex(income_order_Q120))
print()
print("── Distribution Q121 (revenu ménage €) ───────────")
print(users_GG_walk_regular["Q121_label"].value_counts().reindex(income_order_Q121))

In [ ]:
print("── Distribution Q120 (revenu ménage CHF) ────────────")
print(users_GG_walk_regular["Q120_label"].value_counts().reindex(income_order_Q120))
n_q120 = users_GG_walk_regular["Q120_label"].notna().sum()
n_q120_nan = users_GG_walk_regular["Q120_label"].isna().sum()
print(f"\nTotal répondants Q120 : {n_q120} / NaN : {n_q120_nan}")

print()
print("── Distribution Q121 (revenu personnel €) ───────────")
print(users_GG_walk_regular["Q121_label"].value_counts().reindex(income_order_Q121))
n_q121 = users_GG_walk_regular["Q121_label"].notna().sum()
n_q121_nan = users_GG_walk_regular["Q121_label"].isna().sum()
print(f"\nTotal répondants Q121 : {n_q121} / NaN : {n_q121_nan}")

print()
print("── Résumé ───────────────────────────────────────────")
print(f"Q120 uniquement (CHF)  : {users_GG_walk_regular['Q120'].notna().sum()}")
print(f"Q121 uniquement (€)    : {users_GG_walk_regular['Q121'].notna().sum()}")
print(f"Les deux renseignés    : {(users_GG_walk_regular['Q120'].notna() & users_GG_walk_regular['Q121'].notna()).sum()}")
print(f"Aucun renseigné        : {(users_GG_walk_regular['Q120'].isna() & users_GG_walk_regular['Q121'].isna()).sum()}")

In [ ]:
# ─── Diagnostic des NaN ───────────────────────────────────────────────────
print(f"users_GG_walk_regular               : {len(users_GG_walk_regular)}")
print(f"respondants_wave1 total       : {len(respondants_wave1)}")
print()

# Combien de user_stat ont un match dans respondants_wave1 ?
matched = users_GG_walk_regular["user_id_fors"].isin(respondants_wave1["id"])
print(f"Users avec match wave1        : {matched.sum()}")
print(f"Users SANS match wave1        : {(~matched).sum()}  ← pas dans wave1 du tout")
print()

# Parmi ceux qui ont un match, combien ont des NaN dans Q10 ?
user_stat_matched = users_GG_walk_regular[matched]
nan_q10 = user_stat_matched["Q10_1_R"].isna().sum()
print(f"Users avec match mais Q10 NaN : {nan_q10}  ← ont répondu au wave1 mais pas à Q10")

In [ ]:
# ─── Check legs with NaN trip_id ──────────────────────────────────────────────
nan_trip = legs_GG_walk_regular[legs_GG_walk_regular["trip_id"].isna()]

print(f"Legs with NaN trip_id : {len(nan_trip)}")
print("\n── Sample of leg_id with NaN trip_id ───────────────")
print(nan_trip["leg_id"].head(20).to_string())

# ─── Check if those leg_id exist in the trip_journey CSV ──────────────────────
missing_in_csv = nan_trip["leg_id"][~nan_trip["leg_id"].isin(leg_trip_journey["leg_id"])]
present_in_csv = nan_trip["leg_id"][ nan_trip["leg_id"].isin(leg_trip_journey["leg_id"])]

print(f"\nNaN leg_id NOT found in trip_journey CSV : {len(missing_in_csv)}")
print(f"NaN leg_id found in trip_journey CSV     : {len(present_in_csv)}")

In [ ]:
# ─── Profile of legs with NaN trip_id ─────────────────────────────────────────
nan_trip = legs_GG_walk_regular[legs_GG_walk_regular["trip_id"].isna()]

print("── Distance ──────────────────────────────────────────")
print(f"Median length : {nan_trip['length_leg'].median():.0f} m")
print(f"Mean length   : {nan_trip['length_leg'].mean():.0f} m")

print("\n── Purpose ───────────────────────────────────────────")
print(nan_trip["leading_stay_purpose"].value_counts())

print("\n── Source layer ──────────────────────────────────────")
print(nan_trip["source_layer"].value_counts())

In [ ]:
legs_GG_walk_regular.head()

In [ ]:
from IPython.display import display, HTML
display(HTML(legs_GG_walk_regular.drop(columns='geometry').head(10).to_html()))

In [ ]:
print(legs_GG_walk_regular.dtypes.to_string())

In [ ]:
# ─── Reprojection pour le plot ────────────────────────────────────────────────
legs_plot   = legs_GG_walk_regular.to_crs(epsg=2056)
canton_plot = agglo_GG 

fig, ax = plt.subplots(figsize=(10, 10))
fig.suptitle("legs_GG_walk", fontsize=13, fontweight="bold") #legs_GG_walk_regular = same as legs_GG_walk because parametre for regular walker = set to 0

legs_plot.plot(ax=ax, color="#4C72B0", linewidth=0.3, alpha=0.3)
canton_plot.boundary.plot(ax=ax, color='black', linewidth=1.5)
ax.set_axis_off()
plt.tight_layout()
plt.show()

In [ ]:
# # ─── Filtre géographique : exclusion des traces de marche contenues dans le Léman ─
# # Le lac capte parfois des faux legs "marche" (dérive GPS sur l'eau, signal réfléchi
# # sur la surface, etc.). On les retire en testant si le leg entier (toute la
# # LineString, pas seulement début/fin) est contenu dans le polygone du lac.

# lac_leman_path = f'{input_file_path}/network_agreg/GEO_LAC_LEMAN-SHP'  # ← adapter le chemin

# lac_leman = gpd.read_file(lac_leman_path)
# lac_leman = lac_leman.to_crs(operation_crs)

# try:
#     lac_leman_geom = lac_leman.union_all()          # geopandas >= 0.14
# except AttributeError:
#     lac_leman_geom = lac_leman.unary_union           # fallback anciennes versions

# # ─── Détection des legs "marche" contenus dans le lac ─────────────────────────
# legs_proj    = legs_GG_walk_regular.to_crs(lac_leman.crs)
# in_lake_mask = legs_proj.geometry.within(lac_leman_geom).values

# n_removed = int(in_lake_mask.sum())
# print(f"Legs marche contenus dans le Léman : {n_removed} / {len(legs_GG_walk_regular)} "
#       f"({n_removed / len(legs_GG_walk_regular) * 100:.2f}%)")

# # ─── Avant / après ──────────────────────────────────────────────────────────
# legs_before_lake_filter = legs_GG_walk_regular.copy()
# legs_GG_walk_regular    = legs_GG_walk_regular[~in_lake_mask].copy()

# print(f"legs_GG_walk_regular après filtre Léman : {len(legs_GG_walk_regular)} legs")

# # ─── Plot comparatif avant / après ─────────────────────────────────────────
# legs_before_plot = legs_before_lake_filter.to_crs(epsg=2056)
# legs_after_plot  = legs_GG_walk_regular.to_crs(epsg=2056)
# lac_leman_plot   = lac_leman.to_crs(epsg=2056)
# canton_plot      = agglo_GG

# fig, axes = plt.subplots(1, 2, figsize=(18, 10))
# fig.suptitle("legs_GG_walk_regular — avant / après filtre Léman", fontsize=13, fontweight="bold")

# for ax, gdf_plot, title in zip(
#     axes,
#     [legs_before_plot, legs_after_plot],
#     [f"Avant ({len(legs_before_plot)} legs)", f"Après ({len(legs_after_plot)} legs)"],
# ):
#     gdf_plot.plot(ax=ax, color="#4C72B0", linewidth=0.3, alpha=0.3)
#     canton_plot.boundary.plot(ax=ax, color='black', linewidth=1.5)
#     lac_leman_plot.boundary.plot(ax=ax, color='#C44E52', linewidth=1.2)
#     ax.set_title(title, fontsize=11)
#     ax.set_axis_off()

# plt.tight_layout()
# plt.show()


In [ ]:
print(f"CRS legs_GG_walk_regular : {legs_GG_walk_regular.crs}")
print(f"CRS agglo_GG            : {agglo_GG.crs}")
print(f"\nBounds legs_GG_walk_regular : {legs_GG_walk_regular.total_bounds}")
print(f"Bounds agglo_GG            : {agglo_GG.total_bounds}")

# ─── Chercher les géométries hors canton ─────────────────────────────────────
xmin_c, ymin_c, xmax_c, ymax_c = agglo_GG.total_bounds

# Bounding box de chaque leg
bounds = legs_GG_walk_regular.bounds
hors_canton = legs_GG_walk_regular[
    (bounds["minx"] < xmin_c) | 
    (bounds["maxx"] > xmax_c) | 
    (bounds["miny"] < ymin_c) | 
    (bounds["maxy"] > ymax_c)
]

print(f"\nLegs hors bounding box canton : {len(hors_canton)}")
if len(hors_canton) > 0:
    print(f"Bounds des legs hors canton : {hors_canton.total_bounds}")

# EXPORTS

In [ ]:
# ─── Export des legs ──────────────────────────────────
legs_exports = {
    'legs_GG_all'          : legs_GG_all,
    'legs_GG_walk'         : legs_GG_walk,
    'legs_GG_walk_regular' : legs_GG_walk_regular,
}

for name, gdf in legs_exports.items():
    base = os.path.join(output_file_path_PL, name)
    gdf_projected = gdf.to_crs(target_crs)
    gdf_projected.to_file(f"{base}.gpkg",     driver="GPKG")
    gdf_projected.to_parquet(f"{base}.parquet")
    gdf_projected.to_csv(f"{base}.csv",        index=False)
    print(f"{name} exporté → .gpkg / .parquet / .csv ({len(gdf):,} legs)")

In [ ]:
# ─── Export des users ─────────────────────
users_exports = {
    'users_GG'              : users_GG,
    'users_GG_walk'         : users_GG_walk,
    'users_GG_walk_regular' : users_GG_walk_regular,
}

for name, df in users_exports.items():
    # Drop géométrie si présente (les users_GG sont issus de drop_duplicates sur legs)
    df_csv = df.drop(columns='geometry', errors='ignore')
    df_csv.to_csv(os.path.join(output_file_path_PL, f"{name}.csv"), index=False)
    print(f"{name} exporté → .csv ({len(df):,} utilisateurs)")